In [ ]:
! pip install rustbpe

In [ ]:
from pathlib import Path

import numpy as np

DATA_DIR: str = "/kaggle/working/FineWeb-edu_sample_10BT"
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
TOKENIZER_STATE_PATH: str = f"{DATA_DIR}/tokenizer_state.pkl"
TOKENIZED_TRAIN_DATASET_PATH: str = f"{DATA_DIR}/train.bin.gz"
TOKENIZED_VAL_DATASET_PATH: str = f"{DATA_DIR}/val.bin.gz"

EOT_TOKEN: str = "<|EOT|>"
SPECIAL_TOKENS: tuple[str, ...] = (
    EOT_TOKEN,
    "<|user_start|>",
    "<|user_end|>",
    "<|assistant_start|>",
    "<|assistant_end|>",
    "<|python_start|>",
    "<|python_end|>",
    "<|output_start|>",
    "<|output_end|>",
)

TOKENIZER_TRAIN_DOCS: int = 1_500_000
TOKENIZER_BATCH_SIZE: int = 50_000
NUM_VAL_DOCS: int = 2000

TOKENS_DTYPE: type[np.uint16] = np.uint16
NUM_MERGEABLE_RANKS: int = 2**14 - 1
VOCAB_SIZE: int = NUM_MERGEABLE_RANKS + len(SPECIAL_TOKENS)
assert np.iinfo(TOKENS_DTYPE).max >= VOCAB_SIZE - 1

In [ ]:
import gzip
from collections.abc import Iterable
from dataclasses import dataclass

import numpy as np
import rustbpe
import tiktoken
from datasets import IterableDataset


@dataclass
class TokenizerState:
    pat_str: str
    mergeable_ranks: dict[bytes, int]
    special_tokens: dict[str, int]
    eot_token_id: int


def train_tokenizer(
    corpus: Iterable[str], num_mergeable_ranks: int, special_tokens: tuple[str, ...]
) -> TokenizerState:
    rustbpe_tokenizer = rustbpe.Tokenizer()
    rustbpe_tokenizer.train_from_iterator(corpus, vocab_size=num_mergeable_ranks)
    mergeable_ranks: dict[bytes, int] = {
        bytes(key): rank for key, rank in rustbpe_tokenizer.get_mergeable_ranks()
    }
    assert len(mergeable_ranks) == num_mergeable_ranks
    assert special_tokens[0] == EOT_TOKEN

    # Ids continue from the last merge id, in tuple order.
    special_token_ids: dict[str, int] = {
        token: num_mergeable_ranks + offset
        for offset, token in enumerate(special_tokens)
    }
    return TokenizerState(
        pat_str=rustbpe_tokenizer.get_pattern(),
        mergeable_ranks=mergeable_ranks,
        special_tokens=special_token_ids,
        eot_token_id=special_token_ids[EOT_TOKEN],
    )


def build_encoding(state: TokenizerState) -> tiktoken.Encoding:
    return tiktoken.Encoding(
        name="dataset_tokenizer",
        pat_str=state.pat_str,
        mergeable_ranks=state.mergeable_ranks,
        special_tokens=state.special_tokens,
    )


def write_split(
    ds: IterableDataset, output_file: str, state: TokenizerState, batch_size: int
) -> None:
    encoding = build_encoding(state)
    eot_token_id = state.eot_token_id
    with gzip.open(output_file, "wb", compresslevel=4) as f:
        for batch in ds.iter(batch_size=batch_size):
            documents = encoding.encode_ordinary_batch(batch["text"])
            tokens = np.empty(
                sum(len(document) + 1 for document in documents), dtype=TOKENS_DTYPE
            )
            offset = 0
            for document in documents:
                tokens[offset] = eot_token_id  # each document is EOT-prefixed
                tokens[offset + 1 : offset + 1 + len(document)] = document
                offset += len(document) + 1
            f.write(tokens.tobytes())
        f.write(np.array([eot_token_id], dtype=TOKENS_DTYPE).tobytes())


def tokenize_dataset(
    ds: IterableDataset,
    state: TokenizerState,
    num_val_docs: int,
    train_output_path: str,
    val_output_path: str,
    batch_size: int,
) -> None:
    write_split(ds.take(num_val_docs), val_output_path, state, batch_size)
    write_split(ds.skip(num_val_docs), train_output_path, state, batch_size)

In [ ]:
import glob
import pickle

from datasets import load_dataset

files: list[str] = sorted(glob.glob("/kaggle/input/**/*.parquet", recursive=True))

ds = load_dataset("parquet", data_files=files, split="train", streaming=True)

tokenizer_state: TokenizerState = train_tokenizer(
    (document["text"] for document in ds.take(TOKENIZER_TRAIN_DOCS)),
    NUM_MERGEABLE_RANKS,
    SPECIAL_TOKENS,
)
with open(TOKENIZER_STATE_PATH, "wb") as f:
    pickle.dump(tokenizer_state, f, protocol=pickle.HIGHEST_PROTOCOL)

tokenize_dataset(
    ds,
    tokenizer_state,
    NUM_VAL_DOCS,
    train_output_path=TOKENIZED_TRAIN_DATASET_PATH,
    val_output_path=TOKENIZED_VAL_DATASET_PATH,
    batch_size=TOKENIZER_BATCH_SIZE,
)